In [ ]:
# Configure SCBFM_ROOT_DIR and optionally SCBFM_FIGURE_DIR before launching Jupyter.
from pathlib import Path
import os
import sys

_candidates = [Path(os.environ['SCBFM_REPO_DIR'])] if os.environ.get('SCBFM_REPO_DIR') else []
_candidates += [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in _candidates if (p / 'src' / 'main.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open Jupyter inside the checkout or set SCBFM_REPO_DIR.')
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from notebook_setup import ROOT_DIR, OUTPUT_DIR, FIGURE_DIR


In [ ]:
import re
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse

## Part I - Import and reorganization to homogene h5ad structure

In [ ]:
expr_df = pd.read_csv(str(ROOT_DIR / 'datasets/GDSC/drug_response_expr_data.csv'), index_col=0, low_memory=False)
expr_df.index = expr_df.index.astype(str)
print(expr_df.shape)
expr_df.head()

In [ ]:
gene_info = pd.read_csv(str(REPO_ROOT / 'data/bulkformer_gene_info.csv'))
sym2ensg = dict(zip(gene_info["gene_symbol"].astype(str), gene_info["ensg_id"].astype(str)))

symbol_cols = [c for c in expr_df.columns if c in sym2ensg]
ensg_ids = [sym2ensg[c] for c in symbol_cols]

expr_mapped = expr_df[symbol_cols].copy()
expr_mapped.columns = ensg_ids
expr_mapped = expr_mapped.loc[:, ~expr_mapped.columns.duplicated(keep="first")]

n_mapped = len(expr_mapped.columns)
n_unmapped = len(expr_df.columns) - len(symbol_cols)
print(f"Mapped: {n_mapped} / {len(expr_df.columns)}  |  Unmapped: {n_unmapped}")


In [ ]:
adata = ad.AnnData(X=expr_mapped.values.astype("float32"))
adata.obs_names = list(expr_mapped.index)
adata.var_names = list(expr_mapped.columns)
adata

In [ ]:
ic50 = pd.read_csv(str(ROOT_DIR / 'datasets/GDSC/drug_response_prediction_IC50.csv'))
ic50_cell_ids = set(ic50["ModelID"].astype(str))
expr_cell_ids = set(adata.obs_names)
overlap = ic50_cell_ids & expr_cell_ids
print(f"Cell lines in IC50: {len(ic50_cell_ids)}")
print(f"Cell lines in expression: {len(expr_cell_ids)}")
print(f"Overlap: {len(overlap)}")


In [ ]:
adata.write(str(ROOT_DIR / 'datasets/GDSC/gdsc.h5ad'))


## Part II - Statistics

In [ ]:
with open(str(REPO_ROOT / 'data/gene_list.txt')) as f:
    gene_list = [line.strip() for line in f if line.strip()]

In [ ]:
gdsc = ad.read_h5ad(str(ROOT_DIR / 'datasets/GDSC/gdsc.h5ad'))
print(gdsc)

In [ ]:
gene_set = set(map(str, gene_list))
var_names = np.asarray(gdsc.var_names.astype(str))

in_list_mask = np.array([g in gene_set for g in var_names], dtype=bool)
not_in_list_mask = ~in_list_mask
not_in_list_weights = not_in_list_mask.astype(np.float64)

chunk_size = 1000

sum_frac_nonzero = 0.0
sum_frac_reads = 0.0
n_obs_done = 0

for start in range(0, gdsc.n_obs, chunk_size):
    end = min(start + chunk_size, gdsc.n_obs)

    X_chunk = gdsc.X[start:end]

    if sparse.issparse(X_chunk):
        X_chunk = X_chunk.tocsr()

        total_nonzero = np.asarray(X_chunk.getnnz(axis=1)).ravel()
        total_reads = np.asarray(X_chunk.sum(axis=1)).ravel()

        # Same as X_chunk[:, not_in_list_mask].sum(axis=1), but avoids sparse fancy indexing.
        not_in_list_reads = np.asarray(X_chunk @ not_in_list_weights).ravel()

        X_binary = X_chunk.copy()
        X_binary.data = np.ones_like(X_binary.data, dtype=np.float64)
        not_in_list_nonzero = np.asarray(X_binary @ not_in_list_weights).ravel()

        del X_binary

    else:
        X_chunk = np.asarray(X_chunk)

        total_nonzero = (X_chunk > 0).sum(axis=1)
        not_in_list_nonzero = (X_chunk[:, not_in_list_mask] > 0).sum(axis=1)

        total_reads = X_chunk.sum(axis=1)
        not_in_list_reads = X_chunk[:, not_in_list_mask].sum(axis=1)

    frac_nonzero_not_in_list = np.divide(
        not_in_list_nonzero,
        total_nonzero,
        out=np.zeros_like(total_nonzero, dtype=float),
        where=total_nonzero > 0,
    )

    frac_reads_not_in_list = np.divide(
        not_in_list_reads,
        total_reads,
        out=np.zeros_like(total_reads, dtype=float),
        where=total_reads > 0,
    )

    sum_frac_nonzero += frac_nonzero_not_in_list.sum()
    sum_frac_reads += frac_reads_not_in_list.sum()
    n_obs_done += end - start

    if start == 0 or n_obs_done % (10 * chunk_size) == 0 or end == gdsc.n_obs:
        print(f"Processed {n_obs_done:,}/{gdsc.n_obs:,} samples")

    del X_chunk

print("Average portion of non-zero genes NOT in gene_list:",
      sum_frac_nonzero / n_obs_done)

print("Average portion of total reads NOT in gene_list:",
      sum_frac_reads / n_obs_done)


## Part III - filter to gene list

In [ ]:
gene_list = [str(g) for g in gene_list]
gene_index = pd.Index(gdsc.var_names.astype(str))

reorder_idx = gene_index.get_indexer(gene_list)
missing = [g for g, i in zip(gene_list, reorder_idx) if i < 0]
if missing:
    raise ValueError(f"{len(missing)} genes from gene_list are missing in gdsc. First 20: {missing[:20]}")

out_path = str(ROOT_DIR / 'datasets/GDSC/gdsc.h5ad')
chunk_size = 1000

chunks = []

for start in range(0, gdsc.n_obs, chunk_size):
    end = min(start + chunk_size, gdsc.n_obs)
    X_chunk = gdsc.X[start:end, :]

    if sparse.issparse(X_chunk):
        # CSC handles column selection more reliably than CSR on some SciPy builds.
        X_chunk = X_chunk.tocsc()[:, reorder_idx].tocsr()
    else:
        X_chunk = np.asarray(X_chunk)[:, reorder_idx]

    chunk = ad.AnnData(
        X=X_chunk,
        obs=gdsc.obs.iloc[start:end].copy(),
        var=pd.DataFrame(index=pd.Index(gene_list, name=gdsc.var_names.name)),
    )
    chunks.append(chunk)

    print(f"Prepared {end:,}/{gdsc.n_obs:,} samples")

gdsc_aligned = ad.concat(chunks, axis=0, join="inner", merge="same")
gdsc_aligned.var_names = pd.Index(gene_list, name=gdsc.var_names.name)

gdsc_aligned.write_h5ad(out_path)
print("Wrote:", out_path)
print("Shape:", gdsc_aligned.shape)